<H1>
實驗：比較不同模型設定的準確率

目的：從講義的 baseline（sigmoid + SGD，隱藏層 32 個神經元）出發，一次只改一項設定，觀察對測試集準確率的影響。

程式流程：
1. 載入 MNIST，轉成 float32 並除以 255 正規化，輸入形狀維持 (28, 28, 1)。
2. `build()` 函式依參數建立模型：隱藏層大小、activation、optimizer 和 Dropout 比例。
3. `configs` 列出 7 組設定，包含 baseline、換 activation、換 optimizer、加大架構、調整 epoch 與 batch。
4. 迴圈逐一訓練，用 20% 訓練資料當驗證集，最後在測試集上計算準確率。
5. 結果整理成表格 `result`，並保留所有模型，供下一段挑出最佳者。

固定隨機種子（10）確保每組起點一致，比較才公平。


In [ ]:
import numpy as np, pandas as pd, tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = (x_train.astype("float32") / 255).reshape(-1, 28, 28, 1)
x_test = (x_test.astype("float32") / 255).reshape(-1, 28, 28, 1)

def build(hidden, act, opt, dropout=0.0):
    keras.utils.set_random_seed(10)
    m = keras.Sequential([keras.Input(shape=(28, 28, 1)), layers.Flatten()])
    for h in hidden:
        m.add(layers.Dense(h, activation=act))
        if dropout:
            m.add(layers.Dropout(dropout))
    m.add(layers.Dense(10, activation="softmax"))
    m.compile(loss="sparse_categorical_crossentropy", optimizer=opt, metrics=["accuracy"])
    return m

configs = [
    ("0 baseline sigmoid+sgd", (32,), "sigmoid", "sgd", 0.0, 20, 200),
    ("1 改 activation: relu+sgd", (32,), "relu", "sgd", 0.0, 20, 200),
    ("2 改 optimizer: sigmoid+adam", (32,), "sigmoid", "adam", 0.0, 20, 200),
    ("3 relu+adam", (32,), "relu", "adam", 0.0, 20, 200),
    ("4 架構: relu+adam 256", (256,), "relu", "adam", 0.0, 20, 200),
    ("5 架構: 256-128 dropout0.2", (256, 128), "relu", "adam", 0.2, 20, 200),
    ("6 epoch 30 batch 128", (256, 128), "relu", "adam", 0.2, 30, 128),
]

rows, store = [], {}
for name, hid, act, opt, dr, ep, bs in configs:
    m = build(hid, act, opt, dr)
    h = m.fit(x_train, y_train, validation_split=0.2, epochs=ep, batch_size=bs, verbose=0)
    acc = m.evaluate(x_test, y_test, verbose=0)[1]
    rows.append({"設定": name, "測試準確率(%)": round(acc * 100, 2), "參數量": m.count_params()})
    store[name] = (m, h, acc)
    print(name, round(acc * 100, 2))

result = pd.DataFrame(rows)
result

<H1>
最佳模型：訓練曲線、混淆矩陣與 tflite 轉檔

目的：從上一段的結果中找出測試準確率最高的模型，分析它的表現，並轉成樹莓派可以執行的格式。

程式流程：
1. 用 `max()` 選出準確率最高的設定。
2. 畫出訓練與驗證的 Accuracy 和 Loss 曲線，檢查有沒有過擬合。
3. 用 `pd.crosstab` 產生混淆矩陣，觀察哪些數字最容易被搞混。
4. 用 `TFLiteConverter` 把模型轉成 `mnist_model.tflite`。
5. 用 `files.download()` 下載檔案，之後上傳到樹莓派的 Lab8 資料夾，搭配 `predict.py` 做推論。

注意：下載的是最佳模型，不是講義原本的 baseline。

In [ ]:
best = max(store, key=lambda k: store[k][2])
model, hist, acc = store[best]
print("最佳:", best, acc)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(hist.history["accuracy"], label="train")
ax[0].plot(hist.history["val_accuracy"], label="val")
ax[0].set_title("Accuracy"); ax[0].legend()
ax[1].plot(hist.history["loss"], label="train")
ax[1].plot(hist.history["val_loss"], label="val")
ax[1].set_title("Loss"); ax[1].legend()
plt.show()

pred = np.argmax(model.predict(x_test, verbose=0), axis=-1)
print(pd.crosstab(y_test, pred, rownames=["label"], colnames=["predict"]))

converter = tf.lite.TFLiteConverter.from_keras_model(model)
open("mnist_model.tflite", "wb").write(converter.convert())
from google.colab import files
files.download("mnist_model.tflite")